# GESP3 C++

知识目标：掌握数据编码、进制转换、位运算等知识，掌握一维数组、字符串及函数的使用，能够独立使用模拟法、枚举法解决对应的算法问题。

## 3.1 数据编码（原码、反码、补码）  

计算机中的有符号整数由‌符号位‌和‌数值位‌组成。最高位（Most Significant Bit, MSB）为符号位：0 表示正数，1 表示负数。

### 原码 Sign-Magnitude

最直观的二进制表示法。符号位表示正负，其余位表示数值的绝对值。正数‌：符号位为 0，数值位为绝对值的二进制。负数‌：符号位为 1，数值位为绝对值的二进制。

### 反码 Ones' Complement

主要用于从原码到补码的过渡，或在某些旧系统中使用。正数‌：反码与原码‌相同‌。负数‌：符号位‌不变‌，数值位‌按位取反‌（0 变 1，1 变 0）。

### 补码 Two's Complement

现代计算机（包括 C++ 运行环境）中‌实际存储‌整数的形式。它解决了原码和反码中“0 有两种表示”以及“加减法逻辑不统一”的问题。正数‌：补码与原码‌相同‌。负数‌：在‌反码‌的基础上‌加 1‌。

```c
#include <iostream>
#include <bitset>
int main() {
    int a = 5;
    int b = -5;

    // std::bitset 展示的是内存中的补码形式
    std::cout << "5 的补码:  " << std::bitset<8>(a) << std::endl;   // 输出: 00000101
    std::cout << "-5 的补码: " << std::bitset<8>(b) << std::endl;   // 输出: 11111011

    // 验证补码加法: 5 + (-5) 应该等于 0
    int sum = a + b;
    std::cout << "5 + (-5) = " << sum << std::endl;                 // 输出: 0
    
    return 0;
}
```

原码‌：直观，适合人类阅读，但不适合计算机运算。  
反码‌：过渡形态，解决了部分符号问题，但仍有两个零（+0和-0）。  
补码‌：计算机实际存储标准，解决了零的唯一性问题，简化了加减法硬件电路，是现代计算机系统（包括 C++ 程序运行环境）的基础。  

## 3.2 进制转换（二进制、八进制、十进制、十六进制）  

进位计数制（简称进制）是用一组固定符号和统一进位规则表示数值的计数系统‌，核心遵循“逢基数进一”的底层逻辑。  
基数‌：该进制中可使用的不同数码的总个数，比如十进制基数为10，对应0-9共10个符号；二进制基数为2，对应0和1两个符号。  
位权‌：每个数位对应的实际权重，等于基数的n次方，从右往左数位的n值从0开始依次递增。  
进位规则‌：当某一位的数值达到基数时，向高位进1，本位归0。  

R进制转十进制‌：按权展开求和，每一位数码乘以基数R的对应幂次（整数部分从右往左幂次从0开始递增，小数部分从左往右幂次从-1开始递减），再将所有结果相加。

```c
#include<iostream>
using namespace std;
//将R进制字符转换为十进制数字（字符范围限定于0-9和A-Z，共计36个字符）
int r_to_decimal_digit(char c){
    int digit=0;
    if(c>='0'&&c<='9'){
        digit=c-'0';
    }else if(c>='A'&&c<='F'){
        digit=c-'A'+10;
    }else{
        //暂时忽略不处理
    }
    return digit;
}
//将R进制字符串转换为十进制数字
long long r_to_decimal(const string& s,int r){
    long long result=0;
    for(int i=0;i<s.length();++i){//位权展开求和
        char c=s[i];
        int digit=r_to_decimal_digit(c);
        result=result*r+digit;
    }
    return result;
}
int main(){
    cout<<r_to_decimal("ABCDEF",16)<<endl;
    return 0;
}
```

‌十进制转R进制‌：整数部分用「除基取余，逆序排列」，小数部分用「乘基取整，顺序排列」。

```c
#include<iostream>
using namespace std;
//将十进制数字转换为R进制字符（字符范围限定于0-9和A-Z，共计36个字符）
char decimal_to_r_digit(int d){
    char c='\0';
    if(d>=0&&d<=9){
        c=(char)('0'+d);
    }else if(d>=10&&d<=35){
        c=(char)('A'+d-10);
    }else{
        //暂时忽略不处理
    }
    return c;
}
//将十进制数字转换为R进制字符串
string decimal_to_r(long long n,int r){
    string s="";
    while(n>0){//整数部分用「除基取余，逆序排列」，小数部分用「乘基取整，顺序排列」
        int remainder=n%r;
        char c=decimal_to_r_digit(remainder);
        s=c+s;
        n/=r;
    }
    return s;
}
int main(){
    cout<<decimal_to_r(11259375,16)<<endl;
    return 0;
}
```

幂次直接转换‌：若两种进制的基数存在幂次关系（如2^2=4），可直接按对应位数分组转换

```c
#include<iostream>
#include<cmath>
using namespace std;
//将R进制字符转换为十进制数字（字符范围限定于0-9和A-Z，共计36个字符）
int r_to_decimal_digit(char c){
    int digit=0;
    if(c>='0'&&c<='9'){
        digit=c-'0';
    }else if(c>='A'&&c<='F'){
        digit=c-'A'+10;
    }else{
        //暂时忽略不处理
    }
    return digit;
}
//将十进制数字转换为R进制字符（字符范围限定于0-9和A-Z，共计36个字符）
char decimal_to_r_digit(int d){
    char c='\0';
    if(d>=0&&d<=9){
        c=(char)('0'+d);
    }else if(d>=10&&d<=35){
        c=(char)('A'+d-10);
    }else{
        //暂时忽略不处理
    }
    return c;
}
//低幂次转换为高幂次
string convert_from_low_base(string s,int lb,int hb){
    if(s=="0") return "0";//处理特例
    int k=round(log(hb)/log(lb));//计算每组位数k(例如 2->16, k=4)
    s.insert(0,(k-s.size()%k)%k,'0');//左侧补零使长度整除k
    string result;
    for(int i=0;i<s.size();i+=k){//每k位一组转换
        int val=0;
        for(int j=0;j<k;++j){
            val=val*lb+r_to_decimal_digit(s[i+j]);//累加计算组内值
        } 
        result+=decimal_to_r_digit(val);
    }
    size_t pos=result.find_first_not_of('0');//去除前导零
    return (pos!=string::npos)?result.substr(pos):"0";
}
int main(){
    cout<<convert_from_low_base("11010110",2,16)<<endl;//输出D6
    cout<<convert_from_low_base("11010110",2,8)<<endl;//输出326
    return 0;
}
```

```c
#include<iostream>
#include<cmath>
using namespace std;
//将R进制字符转换为十进制数字（字符范围限定于0-9和A-Z，共计36个字符）
int r_to_decimal_digit(char c){
    int digit=0;
    if(c>='0'&&c<='9'){
        digit=c-'0';
    }else if(c>='A'&&c<='F'){
        digit=c-'A'+10;
    }else{
        //暂时忽略不处理
    }
    return digit;
}
//将十进制数字转换为R进制字符（字符范围限定于0-9和A-Z，共计36个字符）
char decimal_to_r_digit(int d){
    char c='\0';
    if(d>=0&&d<=9){
        c=(char)('0'+d);
    }else if(d>=10&&d<=35){
        c=(char)('A'+d-10);
    }else{
        //暂时忽略不处理
    }
    return c;
}
//高幂次转换为低幂次
string convert_from_high_base(string s,int lb,int hb){
    if(s.empty()) return "0";//处理特例
    int k=round(log(hb)/log(lb));//验证幂次关系并计算展开位数k
    string result="";
    bool isLeading=true;//标记是否还在处理前导零部分
    for(char c:s){
        int val=r_to_decimal_digit(c);
        string group="";//将当前位的值转换为k位目标进制字符串
        for(int i=0;i<k;++i){//生成k位字符串
            group=decimal_to_r_digit(val%lb)+group;
            val/=lb;
        }
        if(isLeading){//如果是最高位组，可以去除前导零；后续组必须保留前导零以保持位权对齐
            size_t pos=group.find_first_not_of('0');//去除当前组的前导零
            if(pos!=string::npos){
                result+=group.substr(pos);
                isLeading=false;
            }else{
                continue;//继续等待非零组
            }
        }else{
            result+=group;//如果不是最高位组，直接追加，保留前导零 (如 1 -> 0001)
        }
    }
    if(result.empty()) return "0";//如果结果为空，说明输入全是0
    return result;
}
int main(){ 
    // 示例 1: 十六进制 (16) 转 二进制 (2) -> k=4
    cout<<convert_from_high_base("1A3F",2,16)<<endl;
    // 1 -> 0001 (去前导0->1), A(10) -> 1010, 3 -> 0011, F(15) -> 1111
    // 结果: 1101000111111

    // 示例 2: 八进制 (8) 转 二进制 (2) -> k=3
    cout<<convert_from_high_base("721",2,8)<<endl;
    // 7 -> 111, 2 -> 010, 1 -> 001
    // 结果: 111010001

    // 示例 3: 九进制 (9) 转 三进制 (3) -> k=2
    cout<<convert_from_high_base("120",3,9)<<endl;
    // 120 (base 9) =1*81+2*9+2*9+0*1=81+18=99.
    // 10200 (base 3) =1*81+0*27+2*9+0*3+0*1=81+18=99.
    return 0;
}
```

## 3.3 位运算（与（&）、或（|）、非（~）、异或（^）、左移（<<）、右移(>>)）  

## 3.4 算法的概念与描述（自然语言描述、流程图描述、伪代码描述）  

## 3.5 C++一维数组基本应用 

## 3.6 字符串及其函数  

## 3.7 算法：枚举法  

## 3.8 算法：模拟法  